# Markerless Motion Capture Validation — MediaPipe

This notebook brings together the full validation of **MediaPipe**-based markerless
motion capture against ground truth (force plates and optical motion capture),
reproducing the structure of the original OpenPose results.

Each section computes the metrics for a family of tasks and appends them to
`result_dataframes`; the final cell concatenates everything into a single
results table comparable to the published OpenPose table.

## Imports

In [ ]:
import glob
import numpy as np
import pandas as pd
from utilities import utils
from tasks import cmj, dj, velocity, rjt, hip, nordic, slr, sls
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
%matplotlib inline

def load_task(folder, drop_ul=False, only_ul=False):
    """Glob a keypoints folder, excluding world-coordinate files."""
    files = sorted(p for p in glob.glob(f"data/keypoints/{folder}/*.txt")
                   if "world" not in p.lower())
    if drop_ul:
        files = [p for p in files if "UL" not in p.upper()]
    if only_ul:
        files = [p for p in files if "UL" in p.upper()]
    return files

result_dataframes = []

## Jump Height

Countermovement jumps (CMJ) and drop jumps (DJ), validated against force plates.
CMJ uses height and gravity pixel-to-metric scaling; DJ jump height uses height scaling.

In [ ]:
# --- data ---
cmj_bl = load_task("cmj")                      # adjust if CMJ bl/ul split differs
cmj_data = utils.read_list('cmj_data')
dj_bl = load_task("dj", drop_ul=True)
dj_ul = load_task("dj", only_ul=True)
dj_data = utils.read_list('dj_data')
cmj_subjects = [f"P{i:02d}" for i in range(3, 19)]

In [ ]:
# --- CMJ: assemble jumps, both PTM references ---
# `subjects` here is the PICKLE DATA (cmj_data), not the ID strings.
# assemble_jumps takes ptm as a string and scales internally.
cmj_bl_files = load_task("cmj")          # bilateral CMJ txt files
cmj_ul_files = []                        # populate if you have unilateral CMJ files

all_jumps_h = cmj.assemble_jumps(cmj_bl_files, cmj_ul_files, cmj_data,
                                 ptm='height', trim_first_ul={'P17','P11'})
all_jumps_g = cmj.assemble_jumps(cmj_bl_files, cmj_ul_files, cmj_data,
                                 ptm='gravity', trim_first_ul={'P17','P11'})

cmj.ba_plots(all_jumps_h, title='height PTM')
cmj_h = cmj.get_metrics(all_jumps_h, ptm='height')
cmj_g = cmj.get_metrics(all_jumps_g, ptm='gravity')
pd.concat([cmj_h, cmj_g], ignore_index=True)

In [ ]:
# --- DJ: jump height + flight + contact (height PTM) ---
dj_jh, dj_ft, dj_ct = dj.get_metrics(dj_bl, dj_ul, dj_data, ptm='height')
dj_jh

In [ ]:
result_dataframes.extend([cmj_h, cmj_g, dj_jh])

## Velocity-Based Tests

Back squat and overhead press, mean & peak barbell velocity, validated against
optical motion capture. Both height and barbell pixel-to-metric scaling.

In [ ]:
bsq = load_task("bsq")
ohp = load_task("ohp")
bsq_data = utils.read_list('bsq_data')
ohp_data = utils.read_list('ohp_data')

bsq_h = velocity.get_metrics(bsq, bsq_data, task='bsq', task_label='Back Squat', ptm='height')
bsq_b = velocity.get_metrics(bsq, bsq_data, task='bsq', task_label='Back Squat', ptm='barbell')
ohp_h = velocity.get_metrics(ohp, ohp_data, task='ohp', task_label='Overhead Press', ptm='height')
ohp_b = velocity.get_metrics(ohp, ohp_data, task='ohp', task_label='Overhead Press', ptm='barbell')

result_dataframes.extend([bsq_h, bsq_b, ohp_h, ohp_b])
pd.concat([bsq_h, bsq_b, ohp_h, ohp_b], ignore_index=True)

## Temporal Metrics

Flight time and contact time for the drop jump and the repeated jump test (RJT),
validated against force plates.

In [ ]:
rjt_files = load_task("rjt")
rjt_data = utils.read_list('rjt_data')

# DJ flight + contact (height PTM) already computed above as dj_ft, dj_ct
rjt_metrics = rjt.get_metrics(rjt_files, rjt_data, ptm='height')

result_dataframes.extend([dj_ft, dj_ct, rjt_metrics])
pd.concat([dj_ft, dj_ct, rjt_metrics], ignore_index=True)

## Angular Metrics

Joint-angle range of motion (and angular velocity where applicable) for the
Nordic curl, single-leg squat, hip rotations, and straight-leg raise, validated
against optical motion capture. Angles are scale-invariant, so no pixel-to-metric
scaling is required (PTM Ref = N/A).

In [ ]:
ndc = load_task("nordic")
slr_files = load_task("slr")
slsq = load_task("slsq")
nordic_data = utils.read_list('nordic_data')
slr_data = utils.read_list('slr_data')
sls_data = utils.read_list('sls_data')
hip_data = utils.read_list('hip_data')

nordic_m = nordic.get_metrics(ndc, nordic_data)
slr_m    = slr.get_metrics(slr_files, slr_data)
sls_m    = sls.get_metrics(slsq, sls_data)
hip_m    = hip.get_metrics("data/keypoints/hip", hip_data)

result_dataframes.extend([nordic_m, slr_m, sls_m, hip_m])
pd.concat([nordic_m, slr_m, sls_m, hip_m], ignore_index=True)

## Results Table

The complete MediaPipe validation table, all tasks and metrics combined.

In [ ]:
# concatenate every task's metrics into one table
final = pd.concat(result_dataframes, ignore_index=True)

# tidy column order to match the OpenPose table
cols = ['Metric','Task','PTM Ref','Ground Truth','MAE','Reliability','ICC']
final = final[[c for c in cols if c in final.columns]]
final

In [ ]:
# save to CSV for the writeup
import os
os.makedirs("results", exist_ok=True)
final.to_csv("results/mediapipe_results.csv", index=False)
print("saved results/mediapipe_results.csv  —", len(final), "rows")